# CoverageEvaluator usage

## 1. What this metric measures

Coverage asks how completely generated `output` represents materially important information from authoritative `context` (`context → output`). It requires `context + output`. Optional `input` describes the generation task and is retained for tracing, persistence, and debugging, but is not sent to the Coverage judge.

## 2. Imports

In [ ]:
from idp_eval import (
    CoverageEvaluator,
    EvaluationCase,
    EvaluationFramework,
    create_azure_judge,
)
from idp_eval.judges import AzureJudgeConfig

## 3. Judge configuration

Applications should inject configuration from their own settings/secrets layer. These are placeholders only. `create_gateway_judge(config=...)` works equivalently; evaluators are backend-independent.

In [ ]:
azure_config = AzureJudgeConfig(
    model="your-azure-deployment",
    azure_endpoint="https://your-resource.openai.azure.com",
    tenant_id="your-tenant-id",
    client_id="your-client-id",
    client_secret="your-client-secret",
    api_version="2024-12-01-preview",
    timeout=180,
    proxy_url=None,
    verify_ssl=True,
    reasoning_effort=None,
)
judge = create_azure_judge(config=azure_config)
framework = EvaluationFramework(
    evaluators=[CoverageEvaluator(judge, verbose=True)],
)

## 4. Basic single-output example

Plain strings and structured `dict` / `list` / nested values are accepted. `render_value()` converts structured values into readable judge text; no domain-specific schema is required.

In [ ]:
source = """
Audit logging is mandatory.
Regional hosting must support the US and EU.
Administrator MFA is mandatory.
"""

case = EvaluationCase(
    case_id="coverage-basic-001",
    input="Generate a concise onboarding requirements summary.",
    context=source,
    output={
        "summary": "Audit logging is included.",
        "regions": ["US", "EU"],
    },
)
result = framework.evaluate(case)["coverage"]

## 5. Understanding the result

The judge identifies distinct source items and classifies each item. Python maps fully present to `1.0`, meaningfully partial to `0.5`, and missing to `0.0`; the final score is their deterministic mean. The judge never produces the aggregate score.

In [ ]:
{
    "score": result.score,
    "label": result.label,
    "explanation": result.explanation,
    "details": result.details,
}
result.details["items"]

## 6. Multiple outputs and `evaluation_scope`

A top-level list is one structured output by default because `evaluation_scope='combined'`. Use `individual` to fan out each top-level item, or `both` to run both views. Case IDs do not affect scoring: the combined trace uses `example-001`; item traces use `example-001:0`, `example-001:1`, and `example-001:2`. Combined coverage is not the average of individual coverage because different outputs may collectively cover different source information.

In [ ]:
epics = [
    {"title": "Auditability", "summary": "Add mandatory audit logging."},
    {"title": "Regional hosting", "summary": "Support US and EU hosting."},
    {"title": "Administrator security", "summary": "Require MFA for administrators."},
]

combined_case = EvaluationCase(
    case_id="example-001", input=case.input, context=source,
    output=epics, evaluation_scope="combined",
)
combined_results = framework.evaluate(combined_case)

individual_case = EvaluationCase(
    case_id="example-001", input=case.input, context=source,
    output=epics, evaluation_scope="individual",
)
individual_results = framework.evaluate(individual_case)

both_case = EvaluationCase(
    case_id="example-001", input=case.input, context=source,
    output=epics, evaluation_scope="both",
)
both_results = framework.evaluate(both_case)
both_results["combined"]["coverage"]
both_results["individual"][0]["coverage"]

Return shapes: combined returns `{"coverage": EvaluationResult(...)}`; individual returns `{"combined": None, "individual": [{"coverage": ...}, ...]}`; both returns the combined metric mapping plus the individual list.

## 7. Optional async usage

Jupyter supports top-level `await`. The framework enforces one shared judge-call concurrency limit.

In [ ]:
async_result = await framework.a_evaluate(case, max_concurrency=4)
async_result["coverage"]

## 8. `evaluate_many()`

`evaluate_many()` handles unrelated generation requests. Each case keeps its own `evaluation_scope`, validation, result shape, and input order.

In [ ]:
case_a = EvaluationCase(input="Summarize security requirements.", context=source, output=epics)
case_b = EvaluationCase(input="Summarize one requirement.", context=source, output=epics[:2], evaluation_scope="individual")
cases = [case_a, case_b]
many_results = framework.evaluate_many(cases)
async_many_results = await framework.a_evaluate_many(cases, max_concurrency=4)

## 9. Close resources

In [ ]:
judge.close()